# Importing Libraries for Machine Learning

This code imports the essential libraries required for data processing, machine learning models, evaluation, and saving/loading models.


In [20]:
# %% [code]
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
import joblib
import warnings
warnings.filterwarnings('ignore')

# ================= DATA LOADING =================

This section of the code is responsible for loading the dataset from a specified file path using `pandas.read_csv`.
- If the file is successfully loaded, it prints a success message along with the shape of the dataset (number of rows and columns).
- If there is any issue (e.g., the file is not found or cannot be read), an exception is raised and the error message is displayed.


In [21]:
try:
    df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Trinity Mobility/sensor.csv')
    print("Data loaded successfully")
    print(f"Initial shape: {df.shape}")
except Exception as e:
    raise ValueError(f"Data loading failed: {str(e)}")

Data loaded successfully
Initial shape: (220320, 55)


# ================= DATA CLEANING =================

This section is responsible for cleaning the dataset:
- The column names are stripped of extra spaces and converted to lowercase using `str.strip()` and `str.lower()`.
- A list of expected column names is created, which includes the `timestamp` column, 52 sensor columns (named `sensor_01`, `sensor_02`, ..., `sensor_52`), and the `machine_status` column.
- The dataset is then re-ordered to ensure that the columns match the expected order, using the `df[expected_columns]` syntax.


In [22]:
df.columns = df.columns.str.strip().str.lower()
expected_columns = ['timestamp'] + [f'sensor_{i:02d}' for i in range(52)] + ['machine_status']
df = df[expected_columns]

# Remove Rows with NaN in Target

This section removes rows where the target column (`machine_status`) contains NaN (missing) values:
- The initial number of rows in the dataset is stored in `initial_count`.
- The `dropna()` method is used to remove rows where the `machine_status` column has NaN values.
- The final number of rows after removal is stored in `cleaned_count`.
- The difference between `initial_count` and `cleaned_count` is printed to indicate how many rows were removed due to missing target values.


In [23]:
initial_count = len(df)
df = df.dropna(subset=['machine_status'])
cleaned_count = len(df)
print(f"\nRemoved {initial_count - cleaned_count} rows with NaN target values")


Removed 0 rows with NaN target values


# Fill Remaining NaN Sensor Values

In this step, we handle missing sensor data:
- The `fillna(method='ffill')` function is used to forward fill the NaN values in the sensor columns (columns from `sensor_01` to `sensor_52`).
- If there are still any NaN values after forward filling, `fillna(method='bfill')` is applied to backfill the remaining missing values.
- After cleaning, the remaining missing values (if any) are displayed by using `df.isna().sum()`, which shows the count of missing values per column.


In [24]:
df.iloc[:, 1:-1] = df.iloc[:, 1:-1].fillna(method='ffill').fillna(method='bfill')
print("Missing values after cleaning:")
print(df.isna().sum())

Missing values after cleaning:
timestamp              0
sensor_00              0
sensor_01              0
sensor_02              0
sensor_03              0
sensor_04              0
sensor_05              0
sensor_06              0
sensor_07              0
sensor_08              0
sensor_09              0
sensor_10              0
sensor_11              0
sensor_12              0
sensor_13              0
sensor_14              0
sensor_15         220320
sensor_16              0
sensor_17              0
sensor_18              0
sensor_19              0
sensor_20              0
sensor_21              0
sensor_22              0
sensor_23              0
sensor_24              0
sensor_25              0
sensor_26              0
sensor_27              0
sensor_28              0
sensor_29              0
sensor_30              0
sensor_31              0
sensor_32              0
sensor_33              0
sensor_34              0
sensor_35              0
sensor_36              0
sensor_37          

# ================= TARGET PROCESSING =================

This step cleans the target values (`machine_status`):
- The target column (`machine_status`) is converted to a string type using `astype(str)`.
- Any leading or trailing whitespace in the target values is removed using `str.strip()`.
- All target values are converted to uppercase using `str.upper()` to ensure uniformity in the dataset.


In [25]:
df['machine_status'] = df['machine_status'].astype(str).str.strip().str.upper()

# Remove Any Remaining Invalid Targets (Empty Strings or 'NAN')

In this step, we filter out invalid target values:
- The valid target values are defined as `['BROKEN', 'NORMAL', 'RECOVERING']`.
- The dataset is filtered to include only rows where the `machine_status` column contains one of these valid values using the `isin()` function.
- Any rows with empty strings or 'NAN' values in the `machine_status` column are removed.


In [26]:
valid_targets = ['BROKEN', 'NORMAL', 'RECOVERING']
df = df[df['machine_status'].isin(valid_targets)]

# Encode Targets

In this step, the target values are encoded into numerical values:
- A `LabelEncoder` is instantiated to convert categorical target values into numerical labels.
- The `fit_transform()` method is applied to the `machine_status` column to encode its values.
- The original target class names are stored in `class_names`, and a mapping between the numeric labels and the original class names is printed to show the final encoded target values.


In [27]:
# Encode targets
le = LabelEncoder()
df['machine_status'] = le.fit_transform(df['machine_status'])
class_names = le.classes_
print("\nFinal target classes:", dict(zip(range(len(class_names)), class_names)))


Final target classes: {0: 'BROKEN', 1: 'NORMAL', 2: 'RECOVERING'}


# Verify Class Distribution

This step verifies the distribution of the target classes:
- The `value_counts()` method is applied to the `machine_status` column to count the occurrences of each unique class (target).
- The distribution of the target classes is then printed to show how balanced or imbalanced the classes are.


In [28]:
# Verify class distribution
class_dist = df['machine_status'].value_counts()
print("\nClass distribution:")
print(class_dist)


Class distribution:
machine_status
1    205836
2     14477
0         7
Name: count, dtype: int64


# Remove Classes with Insufficient Samples

In this step, we filter out classes that have too few samples:
- The minimum number of samples required per class is set to `min_samples = 2`.
- The `groupby()` method groups the data by `machine_status`, and `transform('count')` counts the number of occurrences of each class.
- Only classes with a count greater than or equal to `min_samples` are retained in the dataset.
- The final class distribution after filtering is printed to show how many classes remain.


In [29]:
# Remove classes with insufficient samples
min_samples = 2
df = df[df.groupby('machine_status')['machine_status'].transform('count') >= min_samples]
print("\nFinal class distribution after filtering:")
print(df['machine_status'].value_counts())


Final class distribution after filtering:
machine_status
1    205836
2     14477
0         7
Name: count, dtype: int64


# ================= FEATURE EXTRACTION =================

This step handles the extraction of features and the target variable:
- The `timestamp` column is kept separate as a reference (`timestamps`).
- The feature matrix `X` is created by selecting all 52 sensor columns (excluding `timestamp` and `machine_status`).
- The target variable `y` is assigned the `machine_status` column.


In [30]:
timestamps = df['timestamp']
X = df.iloc[:, 1:-1]  # All 52 sensor columns
y = df['machine_status']

# Verify We Have Exactly 52 Features

This step ensures that the feature matrix `X` contains exactly 52 features:
- An `assert` statement is used to check if the number of columns in `X` is equal to 52.
- If the number of features is not 52, an error message is raised with the actual number of features.


In [31]:
# Verify we have exactly 52 features
assert X.shape[1] == 52, f"Expected 52 features, got {X.shape[1]}"

# ================= TRAIN-TEST SPLIT =================

This step splits the data into training and testing sets:
- The `train_test_split()` function is used to divide the feature matrix `X` and target variable `y` into training and test sets.
- 30% of the data is reserved for testing (`test_size=0.3`), and the remaining 70% is used for training.
- The `random_state=42` ensures the split is reproducible.
- The `stratify=y` ensures that the class distribution is preserved in both the training and testing sets.
- The sizes of the training and testing sets are printed to verify the split.


In [32]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)
print(f"\n Train-Test Split:")
print(f"Train: {X_train.shape[0]} samples")
print(f"Test: {X_test.shape[0]} samples")


 Train-Test Split:
Train: 154224 samples
Test: 66096 samples


# ================= FEATURE SCALING =================

In this step, the sensor data is standardized using `StandardScaler`:
- The scaler computes the mean and standard deviation from the training data and scales both training and test sets accordingly.
- `fit_transform()` is applied on `X_train` and `transform()` on `X_test` to avoid data leakage.
- After scaling, `np.nan_to_num()` is used to replace any unexpected `NaN` or infinite values with numerical values (e.g., 0), ensuring clean input for model training.


In [33]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Final NaN check and cleanup
X_train_scaled = np.nan_to_num(X_train_scaled)
X_test_scaled = np.nan_to_num(X_test_scaled)


# ================= MODEL TRAINING =================

In this step, we train and evaluate four machine learning models:
- **Random Forest**, **Logistic Regression**, **K-Nearest Neighbors**, and **Decision Tree** are initialized with predefined parameters.
- Each model is trained using the scaled training data.
- After training, predictions are made on the test set, and performance metrics (accuracy, precision, recall, F1-score) are collected.
- A confusion matrix is plotted for each model to visualize classification results.
- Any model that fails during training or prediction is caught with a try-except block and logged accordingly.


In [ ]:
import matplotlib.pyplot as plt
models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(),
    'Decision Tree': DecisionTreeClassifier(random_state=42)
}

results = []
for name, model in models.items():
    print(f"\n{'='*50}")
    print(f"Training {name}...")

    try:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)

        # Store results
        report = classification_report(y_test, y_pred, output_dict=True)
        results.append({
            'Model': name,
            'Accuracy': report['accuracy'],
            'Precision': report['weighted avg']['precision'],
            'Recall': report['weighted avg']['recall'],
            'F1': report['weighted avg']['f1-score']
        })

        # Plot confusion matrix
        cm = confusion_matrix(y_test, y_pred)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
        disp.plot(cmap='Blues')
        plt.title(f'{name} Confusion Matrix')
        plt.show()

        print(f"{name} trained successfully")
    except Exception as e:
        print(f"{name} failed: {str(e)}")
        continue


Training Random Forest...


# ================= RESULTS & SAVING =================

After training all models, their performance metrics are compiled into a DataFrame and sorted by accuracy.  
This provides a clear comparison to identify the best-performing model.  
The results are then displayed in tabular form using `display()` for readability.


In [ ]:
# ================= RESULTS & SAVING =================
results_df = pd.DataFrame(results).sort_values('Accuracy', ascending=False)
print("\n Model Performance Comparison:")
display(results_df)

# ================= SAVING TRAINED ARTIFACTS =================

The best-performing model (based on accuracy) is selected and saved along with essential components:
- The **trained model**, **scaler**, **label encoder**, **feature order**, and **timestamp column**.
These are bundled and saved using `joblib` to allow for consistent inference later.  
Finally, the pipeline summary is printed to confirm successful execution.


In [ ]:
# Save artifacts
best_model = models[results_df.iloc[0]['Model']]
joblib.dump({
    'model': best_model,
    'scaler': scaler,
    'label_encoder': le,
    'feature_order': list(X.columns),
    'timestamp_column': 'timestamp'
}, 'Tranied_Model.joblib')

print("\n Saved artifacts with:")
print(f"- Best model: {results_df.iloc[0]['Model']}")
print(f"- Cleaned target classes: {list(class_names)}")
print(f"- Timestamp + {len(X.columns)} sensor features")

print("\n Pipeline executed successfully!")